# Day 045 — Exercise 3: transform_record

**What you'll build:** `transform_record(record: dict) -> dict` — clean one valid record: strip whitespace from strings, coerce `amount` to a rounded float, and title-case `region`.

**Why it matters:** Raw CSV data is always strings. The transform step converts values to the types the database expects: `amount` must be a `float` (not the string `'999.99'`), whitespace must be stripped, and values should be normalized so that `'east'` and `'East'` don't create two different groups in a GROUP BY query. `transform_record` is a pure function: it takes a dict and returns a new dict — never mutates the input.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import csv
import io
from sqlalchemy import create_engine, String, Float, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = 'sales'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    date:     Mapped[str]   = mapped_column(String(20))
    product:  Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    amount:   Mapped[float] = mapped_column()
    region:   Mapped[str]   = mapped_column(String(50))

    def __repr__(self):
        return f'Sale(id={self.id}, product={self.product!r}, amount={self.amount})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine

## Your Implementation

In [ ]:
def transform_record(record: dict) -> dict:
    """
    Return a new cleaned dict from a validated raw record.

    Transformations:
      date     → record['date'].strip()
      product  → record['product'].strip()
      category → record.get('category', '').strip()
      amount   → round(float(record['amount']), 2)
      region   → record.get('region', '').strip().title()

    Do NOT mutate the input dict — return a fresh dict.
    Only call this on records that passed validate_record.
    """
    # TODO: return {
    # TODO:     'date':     record['date'].strip(),
    # TODO:     'product':  record['product'].strip(),
    # TODO:     'category': record.get('category', '').strip(),
    # TODO:     'amount':   round(float(record['amount']), 2),
    # TODO:     'region':   record.get('region', '').strip().title(),
    # TODO: }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'transform_record' in globals()
        passed += 1; print('\u2705 Check 1: transform_record is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    raw = {'date': ' 2024-01-15 ', 'product': ' Laptop ',
           'category': ' Electronics ', 'amount': '999.99', 'region': 'east'}

    # Check 2: returns a dict
    try:
        result = transform_record(raw)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a dict')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: amount is float, not string
    try:
        assert isinstance(result['amount'], float), \
            f'amount should be float, got {type(result["amount"]).__name__}'
        assert abs(result['amount'] - 999.99) < 0.001
        passed += 1; print(f'\u2705 Check 3: amount coerced to float ({result["amount"]})')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: strings are stripped
    try:
        assert result['date'] == '2024-01-15', \
            f'date not stripped: {result["date"]!r}'
        assert result['product'] == 'Laptop', \
            f'product not stripped: {result["product"]!r}'
        assert result['category'] == 'Electronics', \
            f'category not stripped: {result["category"]!r}'
        passed += 1; print('\u2705 Check 4: strings are stripped')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: region is title-cased
    try:
        assert result['region'] == 'East', \
            f'region should be title-cased: got {result["region"]!r}'
        passed += 1; print(f'\u2705 Check 5: region title-cased ({result["region"]!r})')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def transform_record(record: dict) -> dict:
    return {
        'date':     record['date'].strip(),
        'product':  record['product'].strip(),
        'category': record.get('category', '').strip(),
        'amount':   round(float(record['amount']), 2),
        'region':   record.get('region', '').strip().title(),
    }
```

</details>